##Setup

In [1]:
!pip install gymnasium shimmy ale-py
!pip install autorom
!AutoROM -y

AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/adventure.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/air_raid.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/alien.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/amidar.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/assault.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asterix.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asteroids.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/atlantis.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/atlantis2.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/backgammon.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/bank_heist.bin
Inst

In [ ]:
from collections import defaultdict, deque
import numpy as np
import cv2
import gymnasium as gym
import ale_py
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# ──────────────────────────────────────────────
# Go-Explore cell functions (pixel-based)
# ──────────────────────────────────────────────
def cellfn(frame):
    cell = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    cell = cv2.resize(cell, (11, 8), interpolation=cv2.INTER_AREA)
    cell = cell // 32
    return cell

def hashfn(cell):
    return hash(cell.tobytes())

# ──────────────────────────────────────────────
# Go-Explore archive
# ──────────────────────────────────────────────
e1 = 0.001
e2 = 0.00001

class Weights:
    times_chosen           = 0.1
    times_chosen_since_new = 0.0
    times_seen             = 0.3

class Powers:
    times_chosen           = 0.5
    times_chosen_since_new = 0.5
    times_seen             = 0.5

class Cell:
    def __init__(self):
        self.times_chosen           = 0
        self.times_chosen_since_new = 0
        self.times_seen             = 0

    def __setattr__(self, key, value):
        object.__setattr__(self, key, value)
        if key != 'score' and hasattr(self, 'times_seen'):
            self.score = self.cellscore()

    def cntscore(self, a):
        w = getattr(Weights, a)
        p = getattr(Powers, a)
        v = getattr(self, a)
        return w / (v + e1) ** p + e2

    def cellscore(self):
        return (self.cntscore('times_chosen') +
                self.cntscore('times_chosen_since_new') +
                self.cntscore('times_seen') + 1)

    def visit(self):
        self.times_seen += 1
        return self.times_seen == 1

    def choose(self):
        self.times_chosen           += 1
        self.times_chosen_since_new += 1
        return self.ram, self.reward, self.trajectory


# ──────────────────────────────────────────────
# Replay buffer
# ──────────────────────────────────────────────
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        states, actions, rewards, next_states, dones = zip(
            *random.sample(self.buffer, batch_size)
        )
        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones))

    def __len__(self):
        return len(self.buffer)


# ──────────────────────────────────────────────
# Network
# ──────────────────────────────────────────────
class DQN(nn.Module):
    def __init__(self, input_shape, n_actions):
        super().__init__()
        self.conv1 = nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)

        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            x = F.relu(self.conv1(dummy))
            x = F.relu(self.conv2(x))
            x = F.relu(self.conv3(x))
            self._feature_size = x.reshape(1, -1).size(1)

        self.fc = nn.Linear(self._feature_size, n_actions)

    def forward(self, x):
        x = x.float() / 255.
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        return self.fc(x.reshape(x.size(0), -1))


# ──────────────────────────────────────────────
# DQN Agent
# ──────────────────────────────────────────────
class DQNAgent:
    def __init__(
        self, env,
        replay_buffer_capacity = 50_000,
        batch_size             = 32,
        gamma                  = 0.99,
        lr                     = 1e-4,
        grad_clip              = 10.0,
        tau                    = 0.005,   # Soft target-update rate
        warmup_steps           = 1_000,   # Steps before any learning
        epsilon_start          = 1.0,
        epsilon_end            = 0.05,
        epsilon_decay_steps    = 50_000,
    ):
        self.env               = env
        self.action_space_size = env.action_space.n
        obs                    = env.observation_space.shape   # (H, W, C)
        self.input_shape       = (obs[2], obs[0], obs[1])      # (C, H, W)

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f'Using device: {self.device}')

        self.policy_net = DQN(self.input_shape, self.action_space_size).to(self.device)
        self.target_net = DQN(self.input_shape, self.action_space_size).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer    = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.memory       = ReplayBuffer(replay_buffer_capacity)
        self.batch_size   = batch_size
        self.gamma        = gamma
        self.grad_clip    = grad_clip
        self.tau          = tau
        self.warmup_steps = warmup_steps

        # Epsilon annealing — linear decay from start → end over decay_steps
        self.epsilon             = epsilon_start
        self.epsilon_start       = epsilon_start
        self.epsilon_end         = epsilon_end
        self.epsilon_decay_steps = epsilon_decay_steps
        self.steps_done          = 0

    def _preprocess(self, state):
        """(H, W, C) numpy → (1, C, H, W) tensor on device."""
        return torch.from_numpy(state).permute(2, 0, 1).unsqueeze(0).to(self.device)

    def select_action(self, state):
        """Epsilon-greedy action selection with linear epsilon decay."""
        # FIX: epsilon is now actually decayed each time select_action is called
        self.steps_done += 1
        self.epsilon = max(
            self.epsilon_end,
            self.epsilon_start - self.steps_done / self.epsilon_decay_steps
        )

        if random.random() < self.epsilon:
            return self.env.action_space.sample()
        with torch.no_grad():
            return self.policy_net(self._preprocess(state)).argmax(1).item()

    def optimize_model(self):
        """One gradient step using Double DQN + Huber loss."""
        if len(self.memory) < max(self.batch_size, self.warmup_steps):
            return None

        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        states      = torch.from_numpy(states).permute(0, 3, 1, 2).to(self.device)
        next_states = torch.from_numpy(next_states).permute(0, 3, 1, 2).to(self.device)
        actions     = torch.from_numpy(actions).long().unsqueeze(1).to(self.device)
        rewards     = torch.from_numpy(rewards).float().unsqueeze(1).to(self.device)
        dones       = torch.from_numpy(dones.astype(np.float32)).unsqueeze(1).to(self.device)

        # Current Q(s, a)
        current_q = self.policy_net(states).gather(1, actions)

        # FIX: Double DQN — policy net picks the action, target net evaluates it.
        # Vanilla DQN's max() uses the same network for both, causing overestimation.
        with torch.no_grad():
            next_actions = self.policy_net(next_states).argmax(1, keepdim=True)
            next_q       = self.target_net(next_states).gather(1, next_actions)
            target_q     = rewards + self.gamma * next_q * (1 - dones)

        # FIX: Huber loss instead of MSE — less sensitive to large TD errors early on
        loss = F.huber_loss(current_q, target_q, delta=1.0)

        self.optimizer.zero_grad()
        loss.backward()
        # FIX: Gradient clipping prevents catastrophic updates from noisy early Q estimates
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), self.grad_clip)
        self.optimizer.step()

        # FIX: Soft target update — τ * policy + (1-τ) * target each step,
        # rather than hard-copying every N steps (which causes instability spikes)
        for p_param, t_param in zip(self.policy_net.parameters(),
                                    self.target_net.parameters()):
            t_param.data.copy_(self.tau * p_param.data + (1 - self.tau) * t_param.data)

        return loss.item()

    def store(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)


# ──────────────────────────────────────────────
# State restoration — never calls env.reset()
# after restoring, which would wipe the state
# ──────────────────────────────────────────────
def restore_env(env, ram):
    """
    Restore ALE state and return the current observation directly.
    Calling env.reset() after ale.restoreState() wipes the restored state,
    so we read the screen via getScreenRGB() instead.
    """
    env.unwrapped.ale.restoreState(ram)
    return env.unwrapped.ale.getScreenRGB()   # (210, 160, 3) uint8


# ──────────────────────────────────────────────
# Hyperparameters
# ──────────────────────────────────────────────
WARMUP_ITERS      = 50    # Pure random Go-Explore before DQN training begins
STEPS_PER_ITER    = 100   # Go-Explore steps per iteration
EXPLORE_EPSILON   = 0.5   # Probability of random action during Go-Explore steps
                          # (separate from the DQN's own epsilon schedule)

archive   = defaultdict(lambda: Cell())
highscore = 0
frames    = 0

env   = gym.make('ALE/MontezumaRevenge-v5', render_mode='rgb_array')
agent = DQNAgent(env)

state, _   = env.reset()
score      = 0
trajectory = []
iterations = 0

restore_cell = None   # Will be set after first iteration

while True:
    found_new_cell = False

    # ── Collect one Go-Explore episode ──────────────────────────────────────
    for step in range(STEPS_PER_ITER):

        # FIX: Separate exploration epsilon from DQN epsilon.
        # During Go-Explore we want a high chance of random actions to find
        # new cells; the DQN's own epsilon schedule governs exploitation separately.
        if iterations < WARMUP_ITERS or random.random() < EXPLORE_EPSILON:
            action = env.action_space.sample()
        else:
            action = agent.select_action(state)

        next_state, reward, terminal, truncated, info = env.step(action)

        # FIX: Detect life loss BEFORE pushing to the replay buffer so the
        # done flag stored in the buffer correctly reflects episode termination.
        life_lost = info['lives'] < 6
        done      = terminal or truncated or life_lost

        # Only train DQN after the warmup period so the buffer has
        # meaningful transitions from Go-Explore's growing archive.
        if iterations >= WARMUP_ITERS:
            agent.store(state, action, reward, next_state, done)
            agent.optimize_model()

        score += reward
        trajectory.append(action)
        frames += 1

        if score > highscore:
            highscore = score

        if done:
            # FIX: Reset score and trajectory on termination so the next
            # archive update doesn't carry over stale cumulative values.
            score      = 0
            trajectory = []
            state, _   = env.reset()
            break
        else:
            # Update archive with the state we just reached
            cell_repr = cellfn(next_state)
            cellhash  = hashfn(cell_repr)
            cell      = archive[cellhash]
            first_visit = cell.visit()

            cell_reward = getattr(cell, 'reward',     -1e9)
            cell_traj   = getattr(cell, 'trajectory', [])
            better  = score > cell_reward
            shorter = score == cell_reward and len(trajectory) < len(cell_traj)

            if first_visit or better or shorter:
                cell.ram        = env.unwrapped.ale.cloneState()
                cell.reward     = score
                cell.trajectory = trajectory.copy()
                cell.times_chosen           = 0
                cell.times_chosen_since_new = 0
                found_new_cell = True

            state = next_state

    # ── Reset times_chosen_since_new on the cell we just explored from ──────
    if found_new_cell and restore_cell is not None:
        restore_cell.times_chosen_since_new = 0

    iterations += 1

    # ── Select next cell to restore from ────────────────────────────────────
    if len(archive) > 0:
        scores = np.array([c.score for c in archive.values()])
        hashes = list(archive.keys())
        probs  = scores / scores.sum()
        chosen_hash = np.random.choice(hashes, p=probs)
        restore_cell = archive[chosen_hash]
        ram, score, trajectory = restore_cell.choose()

        # FIX: Restore state correctly — getScreenRGB() gets the observation
        # without calling env.reset() which would overwrite the restored state
        state = restore_env(env, ram)
    else:
        state, _ = env.reset()
        score      = 0
        trajectory = []

    loss_str = f'{agent.memory.__len__():6d} buf'
    print(
        f"Iter: {iterations:5d} | Cells: {len(archive):5d} | "
        f"Frames: {frames:8d} | MaxReward: {highscore:.1f} | "
        f"Eps: {agent.epsilon:.3f} | {loss_str}"
    )

Using device: cuda
Iter:     1 | Cells:    18 | Frames:       79 | MaxReward: 0.0 | Eps: 1.000 |      0 buf
Iter:     2 | Cells:    18 | Frames:       88 | MaxReward: 0.0 | Eps: 1.000 |      0 buf
Iter:     3 | Cells:    33 | Frames:      174 | MaxReward: 0.0 | Eps: 1.000 |      0 buf
Iter:     4 | Cells:    34 | Frames:      194 | MaxReward: 0.0 | Eps: 1.000 |      0 buf
Iter:     5 | Cells:    34 | Frames:      195 | MaxReward: 0.0 | Eps: 1.000 |      0 buf
Iter:     6 | Cells:    36 | Frames:      201 | MaxReward: 0.0 | Eps: 1.000 |      0 buf
Iter:     7 | Cells:    42 | Frames:      275 | MaxReward: 0.0 | Eps: 1.000 |      0 buf
Iter:     8 | Cells:    43 | Frames:      290 | MaxReward: 0.0 | Eps: 1.000 |      0 buf
Iter:     9 | Cells:    45 | Frames:      366 | MaxReward: 0.0 | Eps: 1.000 |      0 buf
Iter:    10 | Cells:    45 | Frames:      375 | MaxReward: 0.0 | Eps: 1.000 |      0 buf
Iter:    11 | Cells:    45 | Frames:      376 | MaxReward: 0.0 | Eps: 1.000 |      0 buf
It